# F2 · Six ways to encode `occupation` (40 levels), with two models

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compare six strategies for a high-cardinality categorical variable (none, one-hot, one-hot with rare categories
grouped, ordinal, frequency, out-of-fold target encoding) with a logistic regression and a gradient boosting, in cross-validation.

## Dataset

`ds.loans(with_missing=False)`: four numeric columns, `contract_type` (one-hot), `occupation` (40 levels, 6 of which with fewer than 20
rows; each one has a true effect on the risk) and the target `defaulted`. `build(occ_encoder, model)` builds the
complete pipeline: standardized numerics, one-hot contract, `occupation` with the given encoder (or left out if `None`).

## Educational purpose

Understand that the choice of the encoder depends on the **model**: the logistic needs an encoding that makes the effect
linear, boosting can isolate the categories by itself. And quantify, with numbers, the value of target encoding.

## Task

1. Fill the dictionary `encoders` with six entries: `"no occupation"` (`None`), `"one-hot"` (`OneHotEncoder(handle_unknown="ignore")`), `"one-hot rare grouped"` (`OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)`), `"ordinal"` (`OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)`), `"frequency"` (`FrequencyEncoder()`) and `"target OOF"` (`TargetEncoderOOF(m=10)`).
2. For each entry and for each of the two models (`logistic`, `boosting`) compute the mean AUC in cross-validation (`cv`, `scoring="roc_auc"`) of `build(enc, model)`. Save `table` (rows: encoders, columns: `logistic`, `boosting`).
3. Save `best_logistic` and `best_boosting`: the names of the encoders with the highest AUC for each model.
4. Compute `logistic_te_gain`: AUC of the logistic with `target OOF` minus AUC with `no occupation`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from common.transformers import FrequencyEncoder, TargetEncoderOOF

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
NOM, OCC = ["contract_type"], ["occupation"]
X = df[NUM + NOM + OCC]
y = df["defaulted"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def build(occ_encoder, model):
    parts = [("num", StandardScaler(), NUM), ("nom", OneHotEncoder(handle_unknown="ignore"), NOM)]
    if occ_encoder is not None:
        parts.append(("occ", occ_encoder, OCC))
    return make_pipeline(ColumnTransformer(parts, sparse_threshold=0), model)


MODELS = {"logistic": lambda: LogisticRegression(max_iter=2000),
          "boosting": lambda: HistGradientBoostingClassifier(max_iter=100, random_state=0)}
print("occupations with fewer than 20 rows:", int((X.occupation.value_counts() < 20).sum()))

## Your work

In [ ]:
encoders = {
    "no occupation": ...,
    "one-hot": ...,
    "one-hot rare grouped": ...,
    "ordinal": ...,
    "frequency": ...,
    "target OOF": ...,
}

In [ ]:
table = pd.DataFrame(index=list(encoders), columns=list(MODELS), dtype=float)
for enc_name, enc in encoders.items():
    for model_name, make in MODELS.items():
        ...
table.round(3)

In [ ]:
best_logistic = ...
best_boosting = ...
logistic_te_gain = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Inside the double loop: `table.loc[enc_name, model_name] = cross_val_score(build(enc, make()), X, y, cv=cv, scoring="roc_auc").mean()`. `make` is a function that returns a new model: call it with `make()`.'):
    check(list(encoders) == ["no occupation", "one-hot", "one-hot rare grouped", "ordinal", "frequency", "target OOF"] and encoders["no occupation"] is None,
          "six encoders, the first one is None", "respect the order and the names required")
    check(table.shape == (6, 2) and table.notna().all().all(), "complete 6 x 2 table", "rows: encoders, columns: logistic, boosting")
    check(table.loc["ordinal", "logistic"] < table.loc["one-hot", "logistic"], "with the logistic the ordinal is worse than the one-hot",
          "a linear model cannot use an integer code with no order")
    check(table.loc["target OOF", "logistic"] > table.loc["no occupation", "logistic"] + 0.005,
          "out-of-fold target encoding adds useful information for the logistic", "target encoding should improve the AUC compared to not using the occupation")
    check(best_logistic == table["logistic"].idxmax() and best_boosting == table["boosting"].idxmax(),
          "best encoders consistent with the table", "use idxmax() on the columns of the table")
    check(close(logistic_te_gain, table.loc["target OOF", "logistic"] - table.loc["no occupation", "logistic"]),
          "target encoding gain computed", "gain = AUC(target OOF) - AUC(no occupation)")
    check(best_logistic != "ordinal" and best_boosting != "no occupation", "the best encoders are not the two trivial cases", "review the table")
done()

---
The commented solution is in `solutions/F_compare/F2_encoders_for_high_cardinality.ipynb`. Open it after you pass the check, or after you have really tried.